# HuggingFace `pipeline`: Inferencia con Transformers Preentrenados
### IPD434 — Computación Evolutiva e Inteligencia Artificial
**Universidad Técnica Federico Santa María**

---

Los **Transformers** son hoy la arquitectura dominante en el Procesamiento de Lenguaje Natural (NLP) y, cada vez más, en visión y audio. En este laboratorio damos el primer paso práctico: usar la API `pipeline` de la librería [`transformers`](https://huggingface.co/docs/transformers) de **HuggingFace** para resolver tareas de NLP con modelos **preentrenados**, sin escribir una sola línea de entrenamiento.

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Usar** la API `pipeline` de HuggingFace `transformers` para inferencia.
2. **Aplicar** modelos preentrenados a tareas de NLP sin entrenamiento adicional.
3. **Interpretar** las salidas de un `pipeline` y seleccionar la tarea adecuada.

> 🧪 **Laboratorio:** primer contacto práctico con el ecosistema HuggingFace.

> 💡 **Prerrequisitos:** [04_RedesNeuronales](../../04_RedesNeuronales/04_RedesNeuronales.ipynb).

## 1. ¿Por qué Transformers? De las RNN a la atención

Antes de los Transformers, las tareas de secuencia (traducción, análisis de sentimiento) se resolvían con **Redes Neuronales Recurrentes (RNN)** y sus variantes (LSTM, GRU). Estas redes procesan el texto **token a token**, arrastrando un estado oculto. Ese diseño tiene dos limitaciones importantes:

- **Dependencias de largo alcance:** la información del primer token debe "sobrevivir" a través de muchos pasos hasta el final; en la práctica se diluye (*vanishing gradient*).
- **Cómputo secuencial:** al depender cada paso del anterior, el entrenamiento **no se puede paralelizar** bien a lo largo de la secuencia.

El **mecanismo de atención** (*attention*) resuelve ambos problemas: permite que cada token "mire" directamente a **todos** los demás y pondere cuáles son relevantes, sin importar la distancia.

<p align="center"><img src="Images/rnn-attention.png" width="480"></p>

### Scaled dot-product attention

La atención se calcula a partir de tres matrices —*queries* $Q$, *keys* $K$ y *values* $V$— obtenidas por proyección lineal de las representaciones de entrada:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

El producto $Q K^{\top}$ mide la **similitud** entre cada query y cada key; se escala por $\sqrt{d_k}$ (la dimensión de las keys) para estabilizar los gradientes, y `softmax` lo convierte en pesos que promedian los *values*.

<p align="center"><img src="Images/dot-product-attention.png" width="360"></p>

### Multi-head attention

En lugar de una sola atención, el Transformer usa **varias cabezas** ($h$) en paralelo, cada una con sus propias proyecciones, y concatena los resultados:

$$\mathrm{MultiHead}(Q,K,V) = \mathrm{Concat}(\mathrm{head}_1, \ldots, \mathrm{head}_h)\, W^{O}, \qquad \mathrm{head}_i = \mathrm{Attention}(Q W_i^{Q},\, K W_i^{K},\, V W_i^{V})$$

Cada cabeza aprende a atender a un tipo distinto de relación (sintáctica, semántica, de correferencia, etc.). Apilando bloques de atención + capas *feed-forward*, con conexiones residuales y `LayerNorm`, se obtiene la arquitectura Transformer completa.

<p align="center"><img src="Images/transformer-architecture.png" width="340"></p>

> 📄 La arquitectura se introdujo en *"Attention Is All You Need"* (Vaswani et al., 2017).

## 2. El ecosistema HuggingFace

[HuggingFace](https://huggingface.co/) es una plataforma que centraliza **modelos**, **datasets** y **aplicaciones** de código abierto de Machine Learning. Su librería `transformers` ofrece tres niveles de abstracción que usaremos en estos notebooks:

- **`pipeline`**: la API de más alto nivel. Encapsula *tokenización → modelo → post-procesamiento* en una sola llamada. Ideal para inferencia rápida.
- **`AutoTokenizer`**: convierte el texto en los índices numéricos (*tokens*) que el modelo entiende, usando exactamente el mismo esquema con el que fue entrenado.
- **`AutoModel`**: carga la arquitectura y los pesos preentrenados del modelo.

Según cómo usen la atención, los modelos se agrupan en dos grandes familias:

| Familia | Ejemplo | Atención | Casos de uso típicos |
|---------|---------|----------|----------------------|
| **Encoder** | BERT | Bidireccional (ve todo el contexto) | Clasificación, NER, extracción |
| **Decoder** | GPT | Causal (solo el pasado) | Generación de texto |

Para empezar solo necesitamos instalar la librería `transformers`.

In [ ]:
# Instalación de la librería transformers de HuggingFace (ejecutar una sola vez)
!pip install transformers

## 3. Tu primer `pipeline`: análisis de sentimiento

La tarea `"text-classification"` (alias de `"sentiment-analysis"`) recibe una frase y devuelve una etiqueta (`POSITIVE` / `NEGATIVE`) junto con un puntaje de confianza. Como no especificamos un modelo, `pipeline` descarga automáticamente un modelo por defecto afinado para esta tarea (un DistilBERT afinado sobre SST-2). Internamente, el `pipeline`:

1. **Tokeniza** el texto de entrada.
2. Lo pasa por el **modelo** (encoder tipo BERT) para obtener *logits*.
3. Aplica `softmax` y **traduce** el índice de mayor probabilidad a su etiqueta legible.

In [ ]:
# Clasificador de sentimiento con el modelo por defecto de HuggingFace
from transformers import pipeline

classifier = pipeline("text-classification")
classifier("We are very sad about your loss :(")

## 📌 Resumen

| Concepto | Idea clave |
|----------|-----------|
| Limitación de RNN | Cómputo secuencial y dependencias de largo alcance difíciles |
| Atención | Cada token pondera a todos los demás: $\mathrm{softmax}(QK^{\top}/\sqrt{d_k})\,V$ |
| Multi-head | Varias cabezas de atención en paralelo capturan relaciones distintas |
| `pipeline` | API de alto nivel para inferencia en una sola llamada |
| Encoder vs decoder | BERT (clasificar) vs GPT (generar) |

## 🔗 Próximos pasos

- Profundizar en tokenización y *fine-tuning*: [Text Classification Hugging Face](Text%20Classification%20Hugging%20Face.ipynb).
- Reutilizar modelos preentrenados en visión: [Transfer Learning Tensorflow](Transfer%20Learning%20Tensorflow.ipynb).

## 📚 Referencias

- **Vaswani, A. et al. (2017).** *Attention Is All You Need*. NeurIPS. https://arxiv.org/abs/1706.03762
- **Devlin, J. et al. (2019).** *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL. https://arxiv.org/abs/1810.04805
- **HuggingFace.** Documentación de `transformers` — Pipelines. https://huggingface.co/docs/transformers/main_classes/pipelines
- **HuggingFace.** *The Hugging Face Course*. https://huggingface.co/learn/nlp-course